# OPERA flagship — continued pretraining on fresh text, then SFT

Continues the OPERA_MAX flagship (`max_run.json`'s arm, d4608 × 2 layers, 385M) on a **fresh**
FineWeb-Edu pool — documents the flagship never saw, neither its training text nor its held-out
set — then fine-tunes the result for chat. Order:

| part | runtime | what | units |
|---|---|---|---|
| A | **CPU** runtime (cheapest) | build the fresh pool, copy to Drive | a few |
| B | **A100** | continued pretraining (resumable, days) | most of the budget |
| C | L4 is enough | did it help? flagship vs continued, same held-out text | ~5 |
| D | A100 | SFT (xl2) → polish (smol-smoltalk) → probes → publish | ~250 |

Every training step is resumable: after a disconnect, re-run the setup cell and the same training
cell; it continues from the last checkpoint with the same plan. A finished run is never retrained.
The plan (pool size, steps, learning rate) is computed once in §2 and saved to Drive
(`runs/cpt_plan.json`), so every later session uses the same numbers.

**Why 2,048-byte sequences (`CPT_T`):** per byte of text this costs only ~10% more compute than
1,024 (OPERA's cost grows as T log T: one more tree level), plus some padding on documents shorter
than 2,048 (§4 prints the fill factor). In exchange the model trains on positions 1,024–2,048
directly — where chat conversations live (SFT runs at 2,048) and where the 77M length curve began
to drift. Set `CPT_T = 1024` in §2 to keep the original length instead.

## 1. Setup (every session, any runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, math, glob, re, shutil, subprocess
DRIVE = '/content/drive/MyDrive/opera_scale'   # same root as OPERA_Scale / OPERA_MAX
RUNS = f'{DRIVE}/runs'
DDATA = f'{DRIVE}/data'
LOCAL = '/content/data'
REPO = '/content/opera-lm'
os.makedirs(LOCAL, exist_ok=True)
os.environ['OPERA_RUNS'] = RUNS
BRANCH = 'main'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = (f'https://{tok}@github.com/Merna-Khalid/OPERA-LM.git' if tok
       else 'https://github.com/Merna-Khalid/OPERA-LM.git')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --oneline
%cd {REPO}
!pip -q install datasets
assert '--skip-docs' in open('experiments/build_fineweb_bytes.py').read(), 'old builder: push the fresh-pool builder first'
assert '--no-samples' in open('experiments/sft.py').read(), 'old sft.py: push the continued-pretraining options first'
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ''
print('runtime:', gpu or 'CPU')

## 2. The plan (computed once, saved to Drive)

Budget split: SFT and polish get `SFT_UNITS`, builds/evaluations/probes/slack get `RESERVE_UNITS`,
continued pretraining gets the rest. Step time comes from the flagship's own training log (same
bytes per step), scaled by log₂ T for the longer sequences.

Sampling is **with replacement**, so one epoch over a pool of N sequences sees about 63% of them;
`EPOCHS` below 1 (a bigger pool for the same steps) means fewer repeats, at the cost of build time
and Drive space (the pool is stored as 2 bytes per text byte).

Re-running this cell after the plan exists just prints it; set `REPLAN = True` only to start over
**before** any training has happened.

In [ ]:
REPLAN = False
PLAN = f'{RUNS}/cpt_plan.json'
if os.path.exists(PLAN) and not REPLAN:
    P = json.load(open(PLAN))
    print('plan already fixed (set REPLAN=True only before training starts):')
else:
    COMPUTE_UNITS = 1315
    UNITS_PER_H = 6.7          # A100, measured on this account
    SFT_UNITS = 260            # §6: xl2 stage + smol polish + probes
    RESERVE_UNITS = 120        # build, evaluations, slack
    CPT_T = 2048               # or 1024 (see the top of the notebook)
    REWARM = 0.5               # re-warm to half the flagship's peak learning rate
    EPOCHS = 0.8               # passes over the fresh pool (with replacement)
    FILL = 0.80                # expected mean sequence length / CPT_T, corrected after the build
    WARMUP = 2000

    cfg = json.load(open(f'{RUNS}/max_run.json'))
    arm = cfg['arm']
    assert cfg['batch'] * 1024 % CPT_T == 0, 'batch x 1024 must divide by CPT_T'
    batch = cfg['batch'] * 1024 // CPT_T          # same bytes per step as the flagship
    log = open(f'{RUNS}/{arm}.log').read() if os.path.exists(f'{RUNS}/{arm}.log') else ''
    rates = re.findall(r'\(([\d.]+)s, ([\d.]+)s/step\)', log)
    s_flag = float(rates[-1][1]) if rates else cfg['ms'] / 1000
    src = 'flagship log' if rates else 'benchmark estimate'
    sec = s_flag * math.log2(CPT_T) / math.log2(1024)
    cpt_units = COMPUTE_UNITS - SFT_UNITS - RESERVE_UNITS
    steps = int(cpt_units / UNITS_PER_H * 3600 / sec)
    pool_gb = math.ceil(steps * batch / EPOCHS * FILL * CPT_T / 1e8) / 10

    old = f"fineweb_bytes_T1024_{cfg['max_bytes'] / 1e9:g}GB"     # the flagship's pool
    old_meta = json.load(open(f'{DDATA}/{old}.meta.json'))
    skip = old_meta['n_docs']
    new = f'fineweb_bytes_T{CPT_T}_{pool_gb:g}GB_skip{skip}'
    P = dict(arm=arm, flagship_batch=cfg['batch'], flagship_lr_scale=cfg['lr_scale'],
             T=CPT_T, batch=batch, lr_scale=REWARM * cfg['lr_scale'], warmup=WARMUP,
             epochs=EPOCHS, sec_per_step=sec, step_time_source=src,
             units_per_h=UNITS_PER_H, cpt_units=cpt_units, est_steps=steps,
             pool_gb=pool_gb, old_pool=old, skip_docs=skip, pool=new,
             sft_units=SFT_UNITS, reserve_units=RESERVE_UNITS, compute_units=COMPUTE_UNITS)
    json.dump(P, open(PLAN, 'w'), indent=2)
    print('plan written:')
for k in ('arm', 'T', 'batch', 'lr_scale', 'warmup', 'epochs', 'sec_per_step', 'step_time_source',
          'cpt_units', 'est_steps', 'pool_gb', 'old_pool', 'skip_docs', 'pool'):
    print(f'  {k:18s} {P[k]}')
print(f"continued pretraining ~{P['cpt_units'] / P['units_per_h']:.0f} A100-hours; "
      f"pool {P['pool_gb']:g} GB of fresh text ({2 * P['pool_gb']:g} GB on disk)")

## 3. Part A — build the fresh pool (CPU runtime)

Skips the flagship pool's documents (`skip_docs`) and **verifies** the skip: while passing over them,
it replays the flagship pool's seeded held-out draws and checks that every document that pool held
out reappears at the same position with identical bytes. If the FineWeb stream has changed, it
stops before writing any training data. Takes a few hours (streaming); a CPU runtime costs almost
nothing. The pool is then copied to Drive and checked.

In [ ]:
P = json.load(open(f'{RUNS}/cpt_plan.json'))
NEW, OLD = P['pool'], P['old_pool']
files = [f'{NEW}.{s}' for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json')]
need = 2 * P['pool_gb'] + 1
on_drive = all(os.path.exists(f'{DDATA}/{f}') for f in files)
if on_drive:
    print(f'{NEW} is already on Drive: nothing to build')
else:
    free_local = shutil.disk_usage('/content').free / 1e9
    free_drive = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
    print(f'needs ~{need:.0f} GB: local free {free_local:.0f} GB, Drive free {free_drive:.0f} GB')
    assert free_local > need, 'not enough local disk: lower EPOCHS-derived pool size in §2 (REPLAN)'
    if free_drive < need:
        print('WARNING: Drive may not have room for the pool (this reading is approximate) -- '
              'check your Google storage before the copy step')
    for s in ('test.pkl', 'meta.json'):                       # the skip check reads these
        shutil.copy(f'{DDATA}/{OLD}.{s}', f'{LOCAL}/{OLD}.{s}')
    !python -u experiments/build_fineweb_bytes.py --max-bytes {P['pool_gb']}e9 --T {P['T']} \
        --skip-docs {P['skip_docs']} --expect-test {LOCAL}/{OLD}.test.pkl --out-dir {LOCAL}
    assert all(os.path.exists(f'{LOCAL}/{f}') for f in files), 'build failed: see the output above'
    for f in files:
        shutil.copy(f'{LOCAL}/{f}', f'{DDATA}/{f}')
    drive.flush_and_unmount(); drive.mount('/content/drive')
    for f in files:
        assert os.path.getsize(f'{DDATA}/{f}') == os.path.getsize(f'{LOCAL}/{f}'), f'Drive copy incomplete: {f}'
    print('pool on Drive:', NEW)
m = json.load(open(f'{DDATA}/{NEW}.meta.json'))
print(f"{m['n_seqs']:,} sequences, {m['total_tokens'] / 1e9:.2f} GB, mean length "
      f"{m['total_tokens'] / m['n_seqs']:.0f} bytes (fill {m['total_tokens'] / m['n_seqs'] / m['T']:.2f}), "
      f"skip verified: {m.get('skip_verified')}")
assert m.get('skip_verified'), 'the pool was built without the skip check'

## 4. Part B — continued pretraining (A100)

Weights from the flagship, fresh optimizer, a re-warmup to half the flagship's peak learning rate
(`lr_scale`, same bytes per step as the flagship), then a cosine decay over the whole run. Output in
`runs/<arm>_cpt/`, log in `runs/<arm>_cpt.log` (on Drive). Re-run this cell after any disconnect;
it resumes. The guard refuses to launch if the real pool would need more than the budget.

In [ ]:
P = json.load(open(f'{RUNS}/cpt_plan.json'))
NEW = P['pool']
for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json'):
    src_, dst_ = f'{DDATA}/{NEW}.{s}', f'{LOCAL}/{NEW}.{s}'
    if not os.path.exists(dst_) or os.path.getsize(dst_) != os.path.getsize(src_):
        print('copying', s, f'({os.path.getsize(src_) / 1e9:.1f} GB)', flush=True)
        shutil.copy(src_, dst_)
m = json.load(open(f'{LOCAL}/{NEW}.meta.json'))
steps = math.ceil(P['epochs'] * m['n_seqs'] / P['batch'])        # sft.py's own formula
units = steps * P['sec_per_step'] / 3600 * P['units_per_h']
print(f"{steps:,} steps x B{P['batch']} x {P['T']} bytes, lr-scale {P['lr_scale']:.3f}, "
      f"warmup {P['warmup']}: ~{steps * P['sec_per_step'] / 3600:.0f} h, ~{units:.0f} units "
      f"(budget {P['cpt_units']})")
assert units <= 1.05 * P['cpt_units'], (
    f'over budget: lower epochs in the plan (REPLAN before training) -- '
    f"{P['epochs'] * P['cpt_units'] / units:.2f} would fit")
LOG = f"{RUNS}/{P['arm']}_cpt.log"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u experiments/sft.py --arm {P['arm']} \
    --pool {LOCAL}/{NEW} --out-tag cpt --T {P['T']} --batch {P['batch']} --lr-scale {P['lr_scale']} \
    --epochs {P['epochs']} --warmup {P['warmup']} --save-every 500 --no-samples 2>&1 | tee -a {LOG}

In [ ]:
# Watch (any session): progress, measured speed, projected units
P = json.load(open(f'{RUNS}/cpt_plan.json'))
LOG = f"{RUNS}/{P['arm']}_cpt.log"
log = open(LOG).read() if os.path.exists(LOG) else ''
plan_f = f"{RUNS}/{P['arm']}_cpt/sft_plan.json"
total = json.load(open(plan_f))['steps'] if os.path.exists(plan_f) else None
done = [int(s) for s in re.findall(r'step\s+(\d+)\s+loss', log)]
times = [(int(a), float(b)) for a, b in re.findall(r'step\s+(\d+)\s+loss.*?\(([\d.]+)s,', log)]
if done and total:
    last = max(done)
    # speed from the last two log lines of the current session (excludes compile time)
    rate = P['sec_per_step']
    if len(times) >= 2 and times[-1][0] > times[-2][0] and times[-1][1] > times[-2][1]:
        rate = (times[-1][1] - times[-2][1]) / (times[-1][0] - times[-2][0])   # same session
    left_h = (total - last) * rate / 3600
    print(f'step {last:,} / {total:,} ({100 * last / total:.1f}%), {rate:.2f} s/step measured, '
          f'~{left_h:.0f} h (~{left_h * P["units_per_h"]:.0f} units) to go')
print('\n'.join(log.splitlines()[-8:]))

## 5. Part C — did it help? (L4 is enough)

Flagship vs continued model on the **flagship pool's held-out documents**, which neither model ever
trained on (the fresh pool skipped them), scored position by position out to 4,096 bytes. The
1–1024 band is the like-for-like loss; the later bands show what training at 2,048 did for long
context.

In [ ]:
P = json.load(open(f'{RUNS}/cpt_plan.json'))
OLD = P['old_pool']
for s in ('test.pkl',):
    if not os.path.exists(f'{LOCAL}/{OLD}.{s}'):
        shutil.copy(f'{DDATA}/{OLD}.{s}', f'{LOCAL}/{OLD}.{s}')
A, B = P['arm'], f"{P['arm']}_cpt"
assert [f for f in glob.glob(f'{RUNS}/{B}/*.pt') if not f.endswith('_train_ckpt.pt')], \
    'continued pretraining has not finished yet (only mid-run saves): finish §4 first'
!python experiments/eval_position.py --arms {A} {B} --labels "flagship" "continued ({P['T']}-byte, fresh text)" \
    --test-pkl {LOCAL}/{OLD}.test.pkl --eval-len 4096 --train-len 1024 --max-docs 300 \
    --batch 1 --device cuda --out {RUNS}/position_curve_cpt
from IPython.display import Image, display
if os.path.exists(f'{RUNS}/position_curve_cpt.png'):
    display(Image(f'{RUNS}/position_curve_cpt.png'))

## 6. Part D — SFT on the continued model (A100)

Same two stages as OPERA_MAX §9/§9b, now starting from `<arm>_cpt`: the xl2 chat mixture, then the
smol-smoltalk polish (fixes xl2's learned refusals and over-long replies), batch 48, the same
learning rates. Set `SEC_PER_STEP` from the earlier stage-1 SFT log (time difference of two lines
200 steps apart / 200).

In [ ]:
P = json.load(open(f'{RUNS}/cpt_plan.json'))
SEC_PER_STEP = 2.0        # measured batch-48 SFT step time (replace with your value)
for name in ('chat_bytes_T2048_xl2', 'chat_bytes_T2048'):
    for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json'):
        src_, dst_ = f'{DDATA}/{name}.{s}', f'{LOCAL}/{name}.{s}'
        assert os.path.exists(src_), f'{src_} missing on Drive (OPERA_MAX §9/§9b cache it)'
        if not os.path.exists(dst_) or os.path.getsize(dst_) != os.path.getsize(src_):
            shutil.copy(src_, dst_)
CHAT, SMOL = f'{LOCAL}/chat_bytes_T2048_xl2', f'{LOCAL}/chat_bytes_T2048'
STAGE1_EPOCHS, POLISH_EPOCHS = 1.0, 1.0
LR1, LR2 = 0.3 * 3 ** 0.5, 0.1 * 3 ** 0.5     # batch 48: sqrt rule from sft.py's batch-16 base
cost = 0
for pool, ep in ((CHAT, STAGE1_EPOCHS), (SMOL, POLISH_EPOCHS)):
    n = json.load(open(pool + '.meta.json'))['n_seqs']
    h = math.ceil(ep * n / 48) * SEC_PER_STEP / 3600
    cost += h * P['units_per_h']
    print(f'{os.path.basename(pool)}: {ep} epoch(s) = {math.ceil(ep * n / 48):,} steps, ~{h:.1f} h')
print(f"SFT total ~{cost:.0f} units (budget {P['sft_units']})")
assert cost <= 1.1 * P['sft_units'], 'over the SFT budget: lower STAGE1_EPOCHS'

In [ ]:
# stage 1: the xl2 chat mixture, from the continued model
assert [f for f in glob.glob(f"{RUNS}/{P['arm']}_cpt/*.pt") if not f.endswith('_train_ckpt.pt')], \
    'continued pretraining has not finished yet: SFT starts from its final weights (finish §4 first)'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u experiments/sft.py --arm {P['arm']} \
    --pool {CHAT} --epochs {STAGE1_EPOCHS} --batch 48 --lr-scale {LR1} \
    --init-from {RUNS}/{P['arm']}_cpt --out-tag cpt_sft 2>&1 | tee -a {RUNS}/{P['arm']}_cpt_sft.log

In [ ]:
# stage 2: the smol-smoltalk polish
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u experiments/sft.py --arm {P['arm']} \
    --pool {SMOL} --epochs {POLISH_EPOCHS} --batch 48 --lr-scale {LR2} \
    --init-from {RUNS}/{P['arm']}_cpt_sft --out-tag cpt_sft_polish 2>&1 | tee -a {RUNS}/{P['arm']}_cpt_sft_polish.log

In [ ]:
# probes at a 1,024-byte reply cap; the earlier (pre-continuation) chat models are compared if present
for tag in ('cpt_sft_polish', 'cpt_sft'):
    print(f'\n################ {tag} ################')
    !python experiments/chat_probe.py --arm {P['arm']} --ckpt-dir {RUNS}/{P['arm']}_{tag} --device cuda --max-new 1024
for tag in ('sft', 'sft_polish', 'cpt_sft', 'cpt_sft_polish'):
    f = f"{RUNS}/{P['arm']}_{tag}/probe.json"
    if os.path.exists(f):
        s = json.load(open(f))['summary']
        print(f"{tag:15s}: {s['auto_passed']}/{s['auto_total']} auto-checks, "
              f"{s['stopped']}/{s['total']} stopped, {s['role_leaks']} role leaks")

## 7. Publish the chat model

Pick the folder from the probe table, then export and push (same Hub repo and Space as OPERA_MAX
§10; the new weights replace the old ones there).

In [ ]:
CHAT_DIR = f"{RUNS}/{P['arm']}_cpt_sft_polish"     # or ..._cpt_sft
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_REPO_MAX = 'TheInvestigator/opera-lm-chat-max'
print('exporting:', CHAT_DIR)
!python experiments/export_hf.py --arm {P['arm']} --ckpt-dir {CHAT_DIR} --out /content/export --push {HF_REPO_MAX}